# Classroom Guide — RAG as Tool for Agent and Agent with MCP

This notebook teaches how a traditional RAG pipeline can be turned into a **tool for an agent**, and how that same agent can be extended with **action tools** and later with an **MCP server**.

## What this notebook covers
This notebook progresses in a practical sequence:

1. **Build a simple RAG pipeline**
   - read HR documents from S3
   - chunk them
   - create embeddings
   - retrieve relevant chunks
   - generate a grounded answer

2. **Wrap RAG as a tool for an agent**
   - create a Strands agent
   - route HR knowledge questions into the RAG tool

3. **Add action tools**
   - create a leave-apply tool
   - create a leave-summary tool
   - let the agent choose between knowledge retrieval and action execution

4. **Introduce MCP**
   - build a small MCP server
   - expose leave tools through the MCP server
   - connect the Strands agent to those MCP tools

5. **Add guardrails and observability**
   - block disallowed questions before invocation
   - inspect traces and metrics
   - understand how to make the workflow safer and easier to debug


# Install requried libraries

#### Below code cell installs required Python packages.

In [113]:
#uncomment below line and run if there is any library error issue
! pip install boto3 python-docx PyPDF2

# Load Your Extracted Documents from your S3 bucket

#### Below code cell connects to S3 and defines helper functions for reading documents from the bucket.

In [114]:
import boto3
from io import BytesIO
from docx import Document
import PyPDF2


s3 = boto3.client('s3')
s3 = boto3.client('s3')
BUCKET_NAME = "rag-demo-docs-sri"  #replace with your s3 bucket name


def read_docx_from_s3(key):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    file_stream = BytesIO(response['Body'].read())
    doc = Document(file_stream)
    return "\n".join([para.text for para in doc.paragraphs])


def read_pdf_from_s3(key):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    file_stream = BytesIO(response['Body'].read())
    reader = PyPDF2.PdfReader(file_stream)
    
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    
    return text


def extract_text(file_key):
    if file_key.endswith(".docx"):
        return read_docx_from_s3(file_key)
    elif file_key.endswith(".pdf"):
        return read_pdf_from_s3(file_key)
    return ""


# Load all documents
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
files = [obj['Key'] for obj in response.get('Contents', [])]

documents = {}
for file in files:
    documents[file] = extract_text(file)

print(list(documents.keys()))

['HR-POL-008_Leave_Policy.docx', 'HR-POL-009_Remote_Work_Policy.docx']


# Basic Chunking (without overlap)

#### Below code cell demonstrates **basic chunking without overlap**.

In [115]:
def basic_chunking(text, chunk_size=500):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

'''
Here we are blindly splitting text. ...
This often breaks context and reduces answer quality.
'''

sample_text = list(documents.values())[0]

chunks = basic_chunking(sample_text)

print("Total chunks:", len(chunks))
print(chunks[5])

Total chunks: 8
hall be treated as unauthorized and may attract disciplinary action.
7.2 Misrepresentation of medical certificates or misuse of leave privileges shall constitute misconduct and be dealt with under HR-POL-005.

8. Cross-Policy Dependencies
8.1 Performance Appraisal (HR-POL-006): Excessive unauthorized absence shall impact compliance and behavioral scores.
8.2 Health Benefits (HR-POL-003): Long-term medical leave cases may require insurance claim and disability accommodation linkage.
8.3 Remote Wo


# Chunking (with overlap)

#### Below code cell demonstrates **chunking with overlap**.

In [116]:
def chunk_with_overlap(text, chunk_size=500, overlap=100):
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0, len(text), step):
        chunks.append(text[i:i+chunk_size])
    
    return chunks

'''
Overlap ensures context continuity across chunks, which significantly improves retrieval quality
'''

chunks = chunk_with_overlap(sample_text)

print("Total chunks:", len(chunks))
print(chunks[0])
print("----")
print(chunks[1])

Total chunks: 9

ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only

1. Objective
This policy defines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. in compliance with the applicable State Shops and Establishments Acts, the Maternity Benefit Act, 1961, and other relevant labour laws. The objective is to ensure u
----
 Acts, the Maternity Benefit Act, 1961, and other relevant labour laws. The objective is to ensure uniform application of leave benefits, workforce continuity, and statutory compliance, while providing adequate rest and recovery time to employees.

2. Scope
This policy applies to all confirmed employees across all grades and locations of ABC Solutions Pvt. Ltd. Leave during probation shall be governed by the terms of appointment and the Employee Onboarding Policy (HR-POL-00

# Enterprise-Level Chunking

#### Below code cell demonstrates **section-based or semantic-style chunking**.

In [117]:
def section_based_chunking(text):
    sections = text.split("\n\n")  # split by paragraphs
    
    chunks = []
    for sec in sections:
        if len(sec.strip()) > 100:
            chunks.append(sec.strip())
    
    return chunks

'''
In enterprise systems, 
we chunk based on semantic boundaries like sections, clauses, or policies—not arbitrary character limits.
'''
chunks = section_based_chunking(sample_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 9
ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only


# Adding Metadata...semantic layer + filtering

#### Below code cell creates structured chunk records and adds **metadata** such as source, chunk ID, and department.

In [118]:
chunked_data = []

for file, text in documents.items():
    chunks = chunk_with_overlap(text)
    
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "text": chunk,
            "source": file,
            "chunk_id": i,
            "department": "HR" if "hr" in file.lower() else "General"
        })

len(chunked_data)

'''
We transformed raw documents into structured, context-preserving, metadata-enriched chunks 
ready for embedding and retrieval

Raw Docs → Text → Smart Chunks → Metadata → (Next: Embeddings)
Chunks → Vectors → Searchable Intelligence

'''

'\nWe transformed raw documents into structured, context-preserving, metadata-enriched chunks \nready for embedding and retrieval\n\nRaw Docs → Text → Smart Chunks → Metadata → (Next: Embeddings)\nChunks → Vectors → Searchable Intelligence\n\n'

# Embedding

#### Below code cell defines the embedding function using Amazon Bedrock.

In [132]:
import boto3
import json

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

def get_embedding(text):
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v1",
        body=json.dumps({
            "inputText": text
        })
    )
    
    result = json.loads(response['body'].read())
    return result['embedding']

sample_chunk = chunked_data[0]["text"]

embedding = get_embedding(sample_chunk)

print("Embedding length:", len(embedding))
print(embedding[:10])  # preview

for item in chunked_data[:20]:
    item["embedding"] = get_embedding(item["text"])


Embedding length: 1536
[0.58203125, 0.18359375, -0.25390625, -0.37890625, -0.072265625, 0.30078125, 0.08154296875, 0.000347137451171875, 0.11181640625, 0.08935546875]


# Similarity Function

#### Below code cell defines cosine similarity.

In [133]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Retrieval Function

#### Below code cell defines the main retrieval function.

In [134]:
'''
Instead of searching keywords, we are retrieving based on meaning. 
Even if wording changes, relevant content is still found.
'''
def retrieve_top_k(query, k=3):
    query_embedding = get_embedding(query)
    
    scores = []
    
    for item in chunked_data:
        if "embedding" not in item:
            continue
        
        score = cosine_similarity(query_embedding, item["embedding"])
        
        scores.append((score, item))
    
    # Sort by highest similarity
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return scores[:k]

#### Below code cell tests the retrieval function with a sample HR query.

In [135]:
results = retrieve_top_k("What is leave policy?")

for score, item in results:
    print("\nScore:", score)
    print("Source:", item["source"])
    print("Text:", item["text"][:300])


Score: 0.6778227627111753
Source: HR-POL-008_Leave_Policy.docx
Text: rned by the terms of appointment and the Employee Onboarding Policy (HR-POL-001). The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).

3. Definitions
3.1 Earned Leave (EL): Leave accrued based on days worked.
3.2 Casual Leave 

Score: 0.6249072217715607
Source: HR-POL-008_Leave_Policy.docx
Text: 
ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only

1. Objective
This policy defines the types, eligibility, accrual, utilization, and administration of leave for e

Score: 0.6248352959428981
Source: HR-POL-008_Leave_Policy.docx
Text: Absence and Misuse
7.1 Absence without approved leave for more than three consecutive working days shall be treated as unauthorized and may attract disciplinary action.
7.2 Misrepresentati

# Metadata Filtering

#### Below code cell creates structured chunk records and adds **metadata** such as source, chunk ID, and department.

In [136]:
def retrieve_with_filter(query, department=None, k=3):
    query_embedding = get_embedding(query)
    
    scores = []
    
    for item in chunked_data:
        if "embedding" not in item:
            continue
        
        if department and item["department"] != department:
            continue
        
        score = cosine_similarity(query_embedding, item["embedding"])
        scores.append((score, item))
    
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return scores[:k]

#### Below code cell tests metadata-aware retrieval.

In [137]:
'''
This is the semantic layer—combining vector search with business filters like department or document type
'''
results = retrieve_with_filter("leave policy", department="HR")

for score, item in results:
    print("\n", score, item["source"])


 0.6521293529722348 HR-POL-008_Leave_Policy.docx

 0.6378384879034483 HR-POL-008_Leave_Policy.docx

 0.6123830320846446 HR-POL-008_Leave_Policy.docx


#### Below code cell is part of the notebook workflow.

In [138]:
'''
Achieved output
Query → Embedding → Similarity Search → Top Relevant Chunks
S3 → Extraction → Chunking → Embeddings → Retrieval

'''

'\nAchieved output\nQuery → Embedding → Similarity Search → Top Relevant Chunks\nS3 → Extraction → Chunking → Embeddings → Retrieval\n\n'

# RAG (LLM + Context Injection)
'''
Retrieved chunks → LLM → Final Answer
'''

#### Below code cell builds the context block from retrieved chunks.

In [139]:
def build_context(results):
    context = ""
    
    for r in results:
        context += f"\n[Source: {r['source']}]\n"
        context += r["text"] + "\n"
    
    return context

#### Below code cell sends the retrieved context and the user query to the Bedrock model.

In [140]:
def generate_response(query, context):
    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps({
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": f"""
You are an enterprise AI assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}
"""
                        }
                    ]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 300,
                "temperature": 0.2
            }
        })
    )
    
    result = json.loads(response['body'].read())
    return result['output']['message']['content'][0]['text']

# reranking function

#### Below code cell defines an LLM-based reranking step.

In [141]:
def rerank_results_llm(query, results, top_k=3):
    
    # results = [(score, item), ...]
    
    # Step 1: Prepare documents
    docs = ""
    for i, (score, item) in enumerate(results):
        docs += f"\nDocument {i}:\n{item['text']}\n"

    # Step 2: Prompt
    prompt = f"""
You are an expert reranking system.

Given a query and multiple documents, rank the top {top_k} most relevant documents.

Return ONLY the document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Answer format:
[2, 0, 1]
"""

    # Step 3: Call Nova (no approval needed)
    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps({
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 100,
                "temperature": 0
            }
        })
    )

    result = json.loads(response['body'].read())
    output = result['output']['message']['content'][0]['text']

    # Step 4: Extract indices safely
    import re
    indices = list(map(int, re.findall(r'\d+', output)))

    # Step 5: Map back to original results
    reranked = []
    for idx in indices[:top_k]:
        if idx < len(results):
            score, item = results[idx]
            reranked.append({
                "text": item["text"],
                "source": item["source"],
                "score": score
            })

    return reranked

#### Below code cell combines the previous steps into one complete RAG pipeline.

In [142]:
def rag_pipeline(query):
    # Step 1: Retrieve
    results = retrieve_top_k(query, k=3)

    # Step 2: Rerank
    final_results = rerank_results_llm(query, results, top_k=3)
    
    # Step 3: Build context
    context = build_context(final_results)
    
    # Step 4: Generate answer
    answer = generate_response(query, context)
    
    return answer, results

#### Below code cell runs the complete RAG pipeline on a sample question.

In [143]:
query = "What is Eligibility for remote work?"

answer, results = rag_pipeline(query)

print("ANSWER:\n", answer)

ANSWER:
 Eligibility for remote work includes:

1. Minimum of 6 months of confirmed service.
2. Most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).
3. No active disciplinary actions or Performance Improvement Plans (PIP).


# STEP 2 - Wrap HR RAG as a Strands Tool

#### Below code cell prints an intermediate output.

In [144]:
test_answer, test_results = rag_pipeline("What is the leave policy?")
print(test_answer)

The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence, misuse of leave, and cross-policy dependencies. For instance, absence without approved leave for more than three consecutive working days is treated as unauthorized and may attract disciplinary action. Additionally, the policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).


#### Below code cell wraps the RAG pipeline as a **tool**.

In [145]:
def hr_rag_tool(query: str) -> str:
    """
    Wrapper around the existing HR RAG chatbot.
    Returns only the final answer text for the agent.
    """
    result = rag_pipeline(query)

    # Handle tuple-style output: (answer, results)
    if isinstance(result, tuple):
        return str(result[0])

    return str(result)

#### Below code cell prints an intermediate output.

In [146]:
print(hr_rag_tool("What is Eligibility for remote work?"))

Eligibility for remote work includes:

1. Minimum of 6 months of confirmed service.
2. Most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).
3. No active disciplinary actions or Performance Improvement Plans (PIP).

Employees on probation, under PIP, or under disciplinary review are excluded unless explicitly approved by the Chief Human Resources Officer (CHRO).


#### Below code cell imports the Strands agent framework.

In [147]:
from strands import Agent
from strands import tool

#### Below code cell wraps the RAG pipeline as a **tool**.

In [148]:
@tool
def hr_rag_search(query: str) -> str:
    """
    Use the enterprise HR knowledge base to answer policy-related HR questions.
    Use this tool for questions about leave policy, contractor rules, HR documents,
    and other HR knowledge queries.
    """
    result = rag_pipeline(query)

    if isinstance(result, tuple):
        return str(result[0])

    return str(result)

#### Below code cell creates a Strands agent that can use the HR RAG tool.

In [149]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search],
    system_prompt="""
You are an enterprise HR assistant.

RULES:
1. For any HR policy or HR document question, always call hr_rag_search.
2. Never answer HR policy questions from your own knowledge.
3. Use the tool for leave policy, contractor policy, benefits, attendance, and HR process questions.
4. Keep answers concise and grounded in the retrieved enterprise content.
"""
)

#### Below code cell prints an intermediate output.

In [150]:
response = agent("What is Eligibility for remote work?")
print(str(response.message))

<thinking>The User is asking about eligibility for remote work. This is an HR policy question and should be answered using the hr_rag_search tool.</thinking>

Tool #1: hr_rag_search
Eligibility for remote work includes:

1. Minimum of 6 months of confirmed service.
2. Most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).
3. No active disciplinary actions or Performance Improvement Plans (PIP).{'role': 'assistant', 'content': [{'text': 'Eligibility for remote work includes:\n\n1. Minimum of 6 months of confirmed service.\n2. Most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).\n3. No active disciplinary actions or Performance Improvement Plans (PIP).'}], 'metadata': {'usage': {'inputTokens': 658, 'outputTokens': 61, 'totalTokens': 719}, 'metrics': {'latencyMs': 598, 'timeToFirstByteMs': 249}}}


# Step 3: add the leave-apply tool

#### Below code cell creates a simple in-memory leave application function.

In [151]:
#Leave apply function
leave_requests=[]

def apply_leave_logic(employee_id: str, days: int, reason: str = "Not specified") -> str:
    """
    Simple demo leave application logic.
    Stores leave requests in memory.
    """
    record = {
        "employee_id": employee_id,
        "days": days,
        "reason": reason
    }
    
    leave_requests.append(record)
    
    return f"Leave applied successfully for {days} day(s) for employee {employee_id}. Reason: {reason}"

#### Below code cell prints an intermediate output.

In [152]:
print(apply_leave_logic("EMP001", 2, "Personal work"))
print(leave_requests)

Leave applied successfully for 2 day(s) for employee EMP001. Reason: Personal work
[{'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}]


#### Below code cell wraps the leave application logic as a Strands tool.

In [153]:
@tool
def apply_leave(days: int, reason: str = "Not specified", employee_id: str = "EMP001") -> str:
    """
    Apply leave for the current employee.

    Use this tool when the user wants to request or apply leave.
    The 'days' parameter is the number of leave days requested.
    The 'reason' is the leave reason.
    If employee_id is not provided, use EMP001.
    """
    return apply_leave_logic(employee_id, days, reason)

#### Below code cell creates an HR assistant agent with both RAG and leave-apply tools.

In [154]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, apply_leave],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR policy, HR documentation, leave rules, contractor rules, and other HR knowledge questions.
2. Use apply_leave when the user wants to request leave or apply leave.
3. Do not expose your internal reasoning.
4. Do not mention tool names unless explicitly asked.
5. Return only the final answer for the user.
"""
)

#### Below code cell tests the RAG-enabled agent with an HR policy question.

In [155]:
response = agent("What is the leave policy?")
print(response.message)

<thinking>I need to search for the leave policy in the HR knowledge base.</thinking>

Tool #1: hr_rag_search
The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence without approved leave, misrepresentation of medical certificates, and the impact of excessive unauthorized absence on performance appraisal scores. Additionally, it mentions cross-policy dependencies with Health Benefits and Remote Work policies. The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).{'role': 'assistant', 'content': [{'text': 'The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence without approved leave, misrepresentation of medical certificates, and the impact of excessive unauthori

#### Below code cell prints an intermediate output.

In [156]:
response = agent("Apply leave for 2 days for employee EMP001 because of personal work.")
print(response.message)


Tool #2: apply_leave
Your leave request for 2 days due to personal work has been successfully applied for employee EMP001.{'role': 'assistant', 'content': [{'text': 'Your leave request for 2 days due to personal work has been successfully applied for employee EMP001.'}], 'metadata': {'usage': {'inputTokens': 1037, 'outputTokens': 24, 'totalTokens': 1061}, 'metrics': {'latencyMs': 410, 'timeToFirstByteMs': 245}}}


#### Below code cell prints an intermediate output.

In [157]:
print(leave_requests)

[{'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'personal work'}]


# STEP 4 - Multi-step orchestration and memory/state

#### Below code cell adds leave summary logic and exposes it as another tool.

In [158]:
def get_leave_summary_logic(employee_id: str = "EMP001") -> str:
    """
    Summarize leave applied by the employee from in-memory state.
    """
    employee_records = [r for r in leave_requests if r["employee_id"] == employee_id]

    if not employee_records:
        return f"No leave applications found for employee {employee_id}."

    total_days = sum(r["days"] for r in employee_records)

    lines = [f"Employee {employee_id} has applied for a total of {total_days} day(s) of leave."]
    lines.append("Leave history:")

    for i, record in enumerate(employee_records, start=1):
        lines.append(
            f"{i}. {record['days']} day(s) - Reason: {record['reason']}"
        )

    return "\n".join(lines)

#### Below code cell prints an intermediate output.

In [159]:
print(get_leave_summary_logic("EMP001"))

Employee EMP001 has applied for a total of 4 day(s) of leave.
Leave history:
1. 2 day(s) - Reason: Personal work
2. 2 day(s) - Reason: personal work


#### Below code cell adds leave summary logic and exposes it as another tool.

In [160]:
@tool
def get_leave_summary(employee_id: str = "EMP001") -> str:
    """
    Get the leave application summary and total leave days for an employee.

    Use this tool when the user asks:
    - how many days of leave they applied for
    - what their leave history is
    - to show previously applied leave
    """
    return get_leave_summary_logic(employee_id)

#### Below code cell creates a richer HR assistant that can:
- answer policy questions
- apply leave
- summarize leave history

In [161]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, apply_leave, get_leave_summary],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR policy, HR documents, leave rules, remote work rules, and other HR knowledge questions.
2. Use apply_leave when the user wants to request or apply leave.
3. Use get_leave_summary when the user asks how many leave days they applied for, asks for leave history, or asks about previously applied leave.
4. If a user request contains multiple intents, complete all required steps in sequence.
5. Do not expose internal reasoning.
6. Do not mention tool names unless explicitly asked.
7. Return only the final answer for the user.
"""
)

#### Below code cell prints an intermediate output.

In [162]:
response = agent("How many days did I apply?")
print(response.message)

<thinking>I need to get the leave summary for the user. Since the user's employee ID is not provided, I'll use the default 'EMP001'.</thinking>

Tool #1: get_leave_summary
You have applied for a total of 4 days of leave. Your leave history is as follows:
1. 2 days for personal work
2. 2 days for personal work{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 4 days of leave. Your leave history is as follows:\n1. 2 days for personal work\n2. 2 days for personal work'}], 'metadata': {'usage': {'inputTokens': 956, 'outputTokens': 38, 'totalTokens': 994}, 'metrics': {'latencyMs': 492, 'timeToFirstByteMs': 263}}}


#### Below code cell prints an intermediate output.

In [163]:
response = agent("Show my leave history.")
print(response.message)

<thinking>I need to show the leave history for the user. Again, since the user's employee ID is not provided, I'll use the default 'EMP001'.</thinking> 
Tool #2: get_leave_summary
Your leave history is as follows:
1. 2 days for personal work
2. 2 days for personal work{'role': 'assistant', 'content': [{'text': 'Your leave history is as follows:\n1. 2 days for personal work\n2. 2 days for personal work'}], 'metadata': {'usage': {'inputTokens': 1148, 'outputTokens': 25, 'totalTokens': 1173}, 'metrics': {'latencyMs': 471, 'timeToFirstByteMs': 325}}}


#### Below code cell tests the RAG-enabled agent with an HR policy question.

In [164]:
response = agent("Check the leave policy and apply leave for 2 days due to personal work.")
print(response.message)

<thinking>First, I need to check the leave policy using the HR knowledge base. Then, I will apply for 2 days of leave due to personal work. Since the user's employee ID is not provided, I'll use the default 'EMP001'.</thinking>

Tool #3: hr_rag_search

Tool #4: apply_leave
The leave policy of ABC Solutions Pvt. Ltd. outlines the different types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave. Your leave application for 2 days due to personal work has been successfully submitted.{'role': 'assistant', 'content': [{'text': 'The leave policy of ABC Solutions Pvt. Ltd. outlines the different types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave. Your leave application for 2 days due to personal work has been successfully submitted.'}], 'metadata': {'usage': {'inputTokens': 1466, 'outputTokens': 58, 'totalTokens': 1524}, 'metrics': {'latencyMs': 594, 'timeToFirstByteMs': 251}}}


#### Below code cell prints an intermediate output.

In [165]:
response = agent("How many days did I apply?")
print(response.message)

<thinking>I need to get the leave summary for the user to see the total leave days applied. Since the user's employee ID is not provided, I'll use the default 'EMP001'.</thinking> 
Tool #5: get_leave_summary
You have applied for a total of 6 days of leave. Your leave history is as follows:
1. 2 days for personal work
2. 2 days for personal work
3. 2 days for personal work{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 6 days of leave. Your leave history is as follows:\n1. 2 days for personal work\n2. 2 days for personal work\n3. 2 days for personal work'}], 'metadata': {'usage': {'inputTokens': 1699, 'outputTokens': 47, 'totalTokens': 1746}, 'metrics': {'latencyMs': 604, 'timeToFirstByteMs': 328}}}


# MCP

#### Below code cell imports FastMCP.

In [166]:
from mcp.server.fastmcp import FastMCP

##  Creating real MCP server for leave tools

#### Below code cell installs required Python packages.

In [71]:
!pip install -U mcp fastapi uvicorn

#### Below code cell prepares a small JSON file that will store leave requests for the MCP server.

In [72]:
'''
#MCP server runs as a separate process, it should not depend on your notebook’s in-memory leave_requests list.
Storing in small json file
'''

import json
import os

LEAVE_DB_PATH = "/home/ec2-user/SageMaker/leave_requests.json"

if not os.path.exists(LEAVE_DB_PATH):
    with open(LEAVE_DB_PATH, "w") as f:
        json.dump([], f)

print("Leave DB ready:", LEAVE_DB_PATH)

Leave DB ready: /home/ec2-user/SageMaker/leave_requests.json


## MCP server script

#### Below code cell imports FastMCP.

In [73]:
mcp_server_code = r'''
import json
import os
from mcp.server.fastmcp import FastMCP

LEAVE_DB_PATH = "/home/ec2-user/SageMaker/leave_requests.json"

mcp = FastMCP("HR Leave MCP Server")


def load_requests():
    if not os.path.exists(LEAVE_DB_PATH):
        return []
    with open(LEAVE_DB_PATH, "r") as f:
        return json.load(f)


def save_requests(data):
    with open(LEAVE_DB_PATH, "w") as f:
        json.dump(data, f, indent=2)


@mcp.tool(description="Apply leave for an employee")
def apply_leave(employee_id: str = "EMP001", days: int = 1, reason: str = "Not specified") -> str:
    requests = load_requests()

    record = {
        "employee_id": employee_id,
        "days": days,
        "reason": reason
    }
    requests.append(record)
    save_requests(requests)

    return f"Leave applied successfully for {days} day(s) for employee {employee_id}. Reason: {reason}"


@mcp.tool(description="Get leave summary and leave history for an employee")
def get_leave_summary(employee_id: str = "EMP001") -> str:
    requests = load_requests()
    employee_records = [r for r in requests if r["employee_id"] == employee_id]

    if not employee_records:
        return f"No leave applications found for employee {employee_id}."

    total_days = sum(r["days"] for r in employee_records)

    lines = [f"Employee {employee_id} has applied for a total of {total_days} day(s) of leave."]
    lines.append("Leave history:")

    for i, record in enumerate(employee_records, start=1):
        lines.append(f"{i}. {record['days']} day(s) - Reason: {record['reason']}")

    return "\\n".join(lines)


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''

#### Below code cell defines the full MCP server script as a Python string.

In [74]:
server_path = "/home/ec2-user/SageMaker/hr_leave_mcp_server.py"

with open(server_path, "w") as f:
    f.write(mcp_server_code)

print("MCP server script written to:", server_path)

MCP server script written to: /home/ec2-user/SageMaker/hr_leave_mcp_server.py


## Start the MCP server

#### Below code cell starts the MCP server as a background process and manages the PID and log files.

In [75]:
import subprocess
import os
import signal
import time

SERVER_SCRIPT = "/home/ec2-user/SageMaker/hr_leave_mcp_server.py"
SERVER_LOG = "/home/ec2-user/SageMaker/hr_leave_mcp_server.log"
SERVER_PID_FILE = "/home/ec2-user/SageMaker/hr_leave_mcp_server.pid"

# Stop older server if one exists
if os.path.exists(SERVER_PID_FILE):
    with open(SERVER_PID_FILE, "r") as f:
        old_pid = int(f.read().strip())
    try:
        os.kill(old_pid, signal.SIGTERM)
        print(f"Stopped old MCP server process: {old_pid}")
    except ProcessLookupError:
        print(f"Old PID {old_pid} not running")
    except Exception as e:
        print(f"Could not stop old process {old_pid}: {e}")

# Start new server
log_file = open(SERVER_LOG, "w")

proc = subprocess.Popen(
    ["python", SERVER_SCRIPT],
    stdout=log_file,
    stderr=log_file,
    start_new_session=True
)

with open(SERVER_PID_FILE, "w") as f:
    f.write(str(proc.pid))

time.sleep(3)

print(f"MCP server started with PID: {proc.pid}")
print(f"Log file: {SERVER_LOG}")

Stopped old MCP server process: 24374
MCP server started with PID: 6703
Log file: /home/ec2-user/SageMaker/hr_leave_mcp_server.log


#### Below code cell checks whether the MCP server process is running.

In [76]:
import os
import signal

with open("/home/ec2-user/SageMaker/hr_leave_mcp_server.pid", "r") as f:
    pid = int(f.read().strip())

try:
    os.kill(pid, 0)
    print(f"Process {pid} is running")
except OSError:
    print(f"Process {pid} is NOT running")

Process 6703 is running


#### Below code cell performs a quick server health check using logs or an HTTP request.

In [77]:
!tail -n 50 /home/ec2-user/SageMaker/hr_leave_mcp_server.log

INFO:     Started server process [6703]
INFO:     Waiting for application startup.
[04/20/26 20:06:34] INFO     StreamableHTTP       ]8;id=523717;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=12495;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\
                             session manager                                    
                             started                                            
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
                                                                                                                                                                                                                                                                                                 

#### Below code cell performs a quick server health check using logs or an HTTP request.

In [78]:
!curl -i http://127.0.0.1:8000/mcp/ || true

HTTP/1.1 307 Temporary Redirect
date: Mon, 20 Apr 2026 20:06:35 GMT
server: uvicorn
content-length: 0
location: ]8;;http://127.0.0.1:8000/mcp\http://127.0.0.1:8000/mcp
]8;;\


##  Connecting Strands agent to MCP server

#### Below code cell imports the MCP client pieces needed to connect Strands to the running MCP server.

In [79]:
from strands import Agent
from strands.tools.mcp import MCPClient

#### Below code cell imports the MCP client pieces needed to connect Strands to the running MCP server.

In [80]:
from mcp.client.streamable_http import streamablehttp_client

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

mcp_client = MCPClient(lambda: streamablehttp_client(MCP_SERVER_URL))
print("MCP client created")

MCP client created


#### Below code cell imports the MCP client pieces needed to connect Strands to the running MCP server.

In [81]:
#Fetching tools from the MCP server
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

mcp_client = MCPClient(lambda: streamablehttp_client(MCP_SERVER_URL))

agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

#### Below code cell prints an intermediate output.

In [82]:
# Testing MCP leave tool routing
response = agent("Apply leave for 2 days due to personal work.")
print(response.message)

<thinking>The user wants to apply for leave. I will use the 'apply_leave' tool to process the request.</thinking>

Tool #1: apply_leave
Your leave request for 2 days due to personal work has been successfully applied.{'role': 'assistant', 'content': [{'text': 'Your leave request for 2 days due to personal work has been successfully applied.'}], 'metadata': {'usage': {'inputTokens': 872, 'outputTokens': 17, 'totalTokens': 889}, 'metrics': {'latencyMs': 401, 'timeToFirstByteMs': 287}}}


#### Below code cell prints an intermediate output.

In [83]:
# Testing MCP leave summary routing

response = agent("How many days did I apply?")
print(response.message)

<thinking>The user wants to know the number of days they applied for leave. I will use the 'get_leave_summary' tool to retrieve the leave summary and then provide the information to the user.</thinking> 
Tool #2: get_leave_summary
You have applied for a total of 2 days of leave for personal work.{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 2 days of leave for personal work.'}], 'metadata': {'usage': {'inputTokens': 1113, 'outputTokens': 17, 'totalTokens': 1130}, 'metrics': {'latencyMs': 387, 'timeToFirstByteMs': 291}}}


#### Below code cell tests the RAG-enabled agent with an HR policy question.

In [84]:
# RAG HR Chatbot - local tool call

response = agent("What is the leave policy?")
print(response.message)

<thinking>The user wants to know the leave policy. I will use the 'hr_rag_search' tool to retrieve the leave policy information.</thinking> 
Tool #3: hr_rag_search
I'm sorry, I couldn't find the leave policy information. Please let me know if you need help with anything else.{'role': 'assistant', 'content': [{'text': "I'm sorry, I couldn't find the leave policy information. Please let me know if you need help with anything else."}], 'metadata': {'usage': {'inputTokens': 1230, 'outputTokens': 28, 'totalTokens': 1258}, 'metrics': {'latencyMs': 534, 'timeToFirstByteMs': 386}}}


#### Below code cell prints an intermediate output.

In [85]:
# leave action, MCP tool
response = agent("Apply leave for 1 day because of a family event.")
print(response.message)

<thinking>The user wants to apply for leave. I will use the 'apply_leave' tool to process the request.</thinking> 
Tool #4: apply_leave
Your leave request for 1 day due to a family event has been successfully applied.{'role': 'assistant', 'content': [{'text': 'Your leave request for 1 day due to a family event has been successfully applied.'}], 'metadata': {'usage': {'inputTokens': 1381, 'outputTokens': 18, 'totalTokens': 1399}, 'metrics': {'latencyMs': 517, 'timeToFirstByteMs': 374}}}


#### Below code cell prints an intermediate output.

In [86]:
# state query, MCP tool
response = agent("Show my leave history.")
print(response.message)

<thinking>The user wants to know their leave history. I will use the 'get_leave_summary' tool to retrieve the leave history information.</thinking> 
Tool #5: get_leave_summary
Here is your leave history:
1. 2 day(s) - Reason: personal work
2. 1 day(s) - Reason: family event
3. 1 day(s) - Reason: personal work
4. 2 day(s) - Reason: personal work
5. 1 day(s) - Reason: family event
6. 2 day(s) - Reason: personal work
7. 1 day(s) - Reason: family event{'role': 'assistant', 'content': [{'text': 'Here is your leave history:\n1. 2 day(s) - Reason: personal work\n2. 1 day(s) - Reason: family event\n3. 1 day(s) - Reason: personal work\n4. 2 day(s) - Reason: personal work\n5. 1 day(s) - Reason: family event\n6. 2 day(s) - Reason: personal work\n7. 1 day(s) - Reason: family event'}], 'metadata': {'usage': {'inputTokens': 1625, 'outputTokens': 104, 'totalTokens': 1729}, 'metrics': {'latencyMs': 1049, 'timeToFirstByteMs': 424}}}


#### Below code cell prints an intermediate output.

In [87]:
#Testing multi-step with MCP included

'''
local tool routing
MCP tool routing
multi-step orchestration together
'''

response = agent("Check the leave policy and apply leave for 1 day due to personal work.")
print(response.message)

<thinking>The user wants to know the leave policy and apply for leave. First, I will use the 'hr_rag_search' tool to retrieve the leave policy information. Then, I will use the 'apply_leave' tool to process the leave application.</thinking> 
Tool #6: hr_rag_search

Tool #7: apply_leave
I'm sorry, I couldn't find the leave policy information. However, your leave request for 1 day due to personal work has been successfully applied.{'role': 'assistant', 'content': [{'text': "I'm sorry, I couldn't find the leave policy information. However, your leave request for 1 day due to personal work has been successfully applied."}], 'metadata': {'usage': {'inputTokens': 1923, 'outputTokens': 34, 'totalTokens': 1957}, 'metrics': {'latencyMs': 1207, 'timeToFirstByteMs': 927}}}


# Agent → local HR RAG tool + MCP tool provider

## Strands hook guardrail

#### Below code cell is part of the notebook workflow.

In [88]:
from strands.hooks.events import BeforeInvocationEvent

#### Below code cell defines a guardrail hook that blocks salary-related questions before the agent or tools are invoked.

In [89]:
## hook guardrail callback
BLOCKED_KEYWORDS = [
    "salary",
    "compensation",
    "payroll",
    "pay band",
    "salary hike",
    "ctc",
    "confidential salary",
    "employee salary"
]

def salary_guardrail(event: BeforeInvocationEvent):
    """
    Strands hook guardrail that blocks salary/compensation related queries
    before the model or tools are invoked.
    """
    # event.messages is the invocation input message list
    combined_text = ""

    for msg in event.messages:
        content = msg.get("content", "")
        if isinstance(content, str):
            combined_text += " " + content
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    combined_text += " " + item["text"]

    combined_text = combined_text.lower()

    if any(keyword in combined_text for keyword in BLOCKED_KEYWORDS):
        raise ValueError(
            "Policy block: I cannot help with salary, compensation, payroll, or confidential pay information."
        )

#### Below code cell creates an agent that can use both:
- the local HR RAG tool
- the MCP-provided leave tools

In [90]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

#### Below code cell defines a guardrail hook that blocks salary-related questions before the agent or tools are invoked.

In [91]:
from strands.hooks import BeforeInvocationEvent

agent.add_hook(salary_guardrail, BeforeInvocationEvent)

#### Below code cell attaches the guardrail to the agent or provides a safe wrapper for calling the agent.

In [92]:
def safe_agent_call(query: str) -> str:
    try:
        response = agent(query)
        return response.message
    except Exception as e:
        return str(e)

#### Below code cell attaches the guardrail to the agent or provides a safe wrapper for calling the agent.

In [93]:
print(safe_agent_call("What is the leave policy?"))


<thinking>The user is asking for the leave policy. I need to use the hr_rag_search tool to find this information.</thinking>

Tool #1: hr_rag_search
<thinking>The tool hr_rag_search did not provide the information. I need to confirm with the user if they want more specific information or if there was a misunderstanding.</thinking>

Could you please provide more details about the type of leave policy you are interested in? For example, are you looking for information on sick leave, vacation leave, or another specific type of leave?{'role': 'assistant', 'content': [{'text': '<thinking>The tool hr_rag_search did not provide the information. I need to confirm with the user if they want more specific information or if there was a misunderstanding.</thinking>\n\nCould you please provide more details about the type of leave policy you are interested in? For example, are you looking for information on sick leave, vacation leave, or another specific type of leave?'}], 'metadata': {'usage': {'in

#### Below code cell attaches the guardrail to the agent or provides a safe wrapper for calling the agent.

In [94]:
print(safe_agent_call("Show me employee salary details."))

Policy block: I cannot help with salary, compensation, payroll, or confidential pay information.


#### Below code cell attaches the guardrail to the agent or provides a safe wrapper for calling the agent.

In [95]:
print(safe_agent_call("what is the salary hike in finance department?."))

Policy block: I cannot help with salary, compensation, payroll, or confidential pay information.


#### Below code cell is part of the notebook workflow.

In [96]:
# Strands Hook guard rails invoke before LLM calls ...similarity with llmguards opensource library
# Mention bedrock gaurdrails post invocation

# Observability and Traceability

#### Below code cell prints an intermediate output.

In [97]:
result = agent("What is the leave policy?")
print(type(result))
print(dir(result))

<thinking>The user is still asking for the leave policy. I need to use the hr_rag_search tool again to find this information.</thinking> 
Tool #2: hr_rag_search
<thinking>The tool hr_rag_search did not provide the information again. I will inform the user that I cannot provide the information at this moment and suggest they reach out to their direct manager or HR department for detailed leave policy information.</thinking>

I'm sorry, but I currently don't have access to the leave policy information. I recommend reaching out to your direct manager or the HR department for detailed information on our leave policies.<class 'strands.agent.agent_result.AgentResult'>
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '

#### Below code cell is part of the **observability and traceability** section.

In [98]:
# safe trace-aware runner
def run_agent_with_trace(query: str):
    try:
        result = agent(query)
        return {
            "ok": True,
            "query": query,
            "message": result.message,
            "traces": getattr(result, "traces", None),
            "raw_result": result
        }
    except Exception as e:
        return {
            "ok": False,
            "query": query,
            "message": str(e),
            "traces": None,
            "raw_result": None
        }

#### Below code cell is part of the **observability and traceability** section.

In [99]:
#trace printer
def print_trace_node(node, indent=0):
    prefix = "  " * indent

    # Try common attributes safely
    name = getattr(node, "name", None) or getattr(node, "span_name", None) or type(node).__name__
    start_time = getattr(node, "start_time", None)
    end_time = getattr(node, "end_time", None)
    duration_ms = None

    if start_time is not None and end_time is not None:
        try:
            duration_ms = (end_time - start_time) * 1000
        except Exception:
            duration_ms = None

    if duration_ms is not None:
        print(f"{prefix}- {name} ({duration_ms:.2f} ms)")
    else:
        print(f"{prefix}- {name}")

    children = getattr(node, "children", None) or []
    for child in children:
        print_trace_node(child, indent + 1)

#### Below code cell is part of the **observability and traceability** section.

In [100]:
# wrapper
def print_agent_traces(trace_result):
    traces = trace_result.get("traces")

    if not traces:
        print("No traces available.")
        return

    print(f"Trace count: {len(traces)}")
    for i, trace in enumerate(traces, start=1):
        print(f"\nTrace #{i}")
        print_trace_node(trace)

#### Below code cell creates an agent that can use both:
- the local HR RAG tool
- the MCP-provided leave tools

In [101]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    callback_handler=None,
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

#### Below code cell is part of the notebook workflow.

In [102]:
def extract_text_from_message(message):
    if isinstance(message, str):
        return message

    if isinstance(message, dict):
        content = message.get("content", [])
        parts = []

        for item in content:
            if isinstance(item, dict) and "text" in item:
                text = item["text"]
                # remove visible thinking blocks if present
                text = text.replace("<thinking>", "").replace("</thinking>", "")
                parts.append(text.strip())

        return "\n".join([p for p in parts if p]).strip()

    return str(message)

#### Below code cell is part of the **observability and traceability** section.

In [103]:
def run_agent_with_trace(query: str):
    try:
        result = agent(query)
        return {
            "ok": True,
            "query": query,
            "message": extract_text_from_message(result.message),
            "traces": getattr(result, "traces", None),
            "metrics": getattr(result, "metrics", None),
            "raw_result": result
        }
    except Exception as e:
        return {
            "ok": False,
            "query": query,
            "message": str(e),
            "traces": None,
            "metrics": None,
            "raw_result": None
        }

#### Below code cell is part of the **observability and traceability** section.

In [104]:
result = agent("What is the leave policy?")
print("Has traces:", hasattr(result, "traces"))
print("Traces value:", getattr(result, "traces", None))
print("Has metrics:", hasattr(result, "metrics"))
print("Metrics value:", getattr(result, "metrics", None))

Has traces: False
Traces value: None
Has metrics: True
Metrics value: EventLoopMetrics(cycle_count=2, tool_metrics={'hr_rag_search': ToolMetrics(tool={'toolUseId': 'tooluse_8JhfuLggZZPiZM8TMt7Tur', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=1, success_count=1, error_count=0, total_time=1.2510321140289307)}, cycle_durations=[1.8858137130737305, 0.6398158073425293], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='7670151c-40c2-487f-87ea-10fdcfa31af9', usage={'inputTokens': 760, 'outputTokens': 53, 'totalTokens': 813}), EventLoopCycleMetric(event_loop_cycle_id='6134778e-60e9-475f-ab86-525abc785236', usage={'inputTokens': 846, 'outputTokens': 62, 'totalTokens': 908})], usage={'inputTokens': 1606, 'outputTokens': 115, 'totalTokens': 1721})], traces=[<strands.telemetry.metrics.Trace object at 0x7fd03aacba40>, <strands.telemetry.metrics.Trace object at 0x7fd03842db50>], accumulated_usage={'inputTokens': 1606, 'out

#### Below code cell is part of the **observability and traceability** section.

In [105]:
def print_metrics(result):
    metrics = getattr(result, "metrics", None)

    if not metrics:
        print("No metrics available.")
        return

    print("=== METRICS ===")

    accumulated_usage = getattr(metrics, "accumulated_usage", None)
    if accumulated_usage:
        print("Token usage:", accumulated_usage)

    cycle_durations = getattr(metrics, "cycle_durations", None)
    if cycle_durations:
        print("Cycle durations:", cycle_durations)
        print("Total execution time (s):", sum(cycle_durations))

    tool_metrics = getattr(metrics, "tool_metrics", None)
    if tool_metrics:
        print("Tool metrics:")
        for tool_name, tool_data in tool_metrics.items():
            print(f"  - {tool_name}: {tool_data}")

#### Below code cell is part of the **observability and traceability** section.

In [106]:
result = agent("Apply leave for 2 days due to personal work.")
print("FINAL ANSWER:")
print(extract_text_from_message(result.message))

print_metrics(result)

FINAL ANSWER:
Your leave for 2 days due to personal work has been applied successfully.
=== METRICS ===
Token usage: {'inputTokens': 3550, 'outputTokens': 179, 'totalTokens': 3729}
Cycle durations: [1.8858137130737305, 0.6398158073425293, 0.5538825988769531, 0.37769317626953125]
Total execution time (s): 3.457205295562744
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_8JhfuLggZZPiZM8TMt7Tur', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=1, success_count=1, error_count=0, total_time=1.2510321140289307)
  - apply_leave: ToolMetrics(tool={'toolUseId': 'tooluse_WduSNZNWBRZtiQ5yhYglPK', 'name': 'apply_leave', 'input': {'reason': 'personal work', 'days': 2}}, call_count=1, success_count=1, error_count=0, total_time=0.01181173324584961)


#### Below code cell is part of the **observability and traceability** section.

In [107]:
def demo_observe(query: str):
    try:
        result = agent(query)

        print("=" * 80)
        print("QUERY:")
        print(query)

        print("\nFINAL ANSWER:")
        print(extract_text_from_message(result.message))

        print("\nOBSERVABILITY:")
        print_metrics(result)

        traces = getattr(result, "traces", None)
        print("\nRAW TRACES AVAILABLE:", bool(traces))
        print("=" * 80)

    except Exception as e:
        print("=" * 80)
        print("QUERY:")
        print(query)
        print("\nBLOCKED / ERROR:")
        print(str(e))
        print("=" * 80)

#### Below code cell is part of the **observability and traceability** section.

In [108]:
demo_observe("What is the leave policy?")


QUERY:
What is the leave policy?

FINAL ANSWER:
The tool did not have the required information again. I will ask the user if they would like to ask another question or provide more information.

I apologize, but I still couldn't find the leave policy information in our knowledge base. Can you provide more details or ask another question?

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 5731, 'outputTokens': 295, 'totalTokens': 6026}
Cycle durations: [1.8858137130737305, 0.6398158073425293, 0.5538825988769531, 0.37769317626953125, 1.8279008865356445, 0.6863229274749756]
Total execution time (s): 5.971429109573364
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_TKWuL3egE2urqaViSmquYe', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=2, success_count=2, error_count=0, total_time=2.47125506401062)
  - apply_leave: ToolMetrics(tool={'toolUseId': 'tooluse_WduSNZNWBRZtiQ5yhYglPK', 'name': 'apply_leave', 'input': {'reas

#### Below code cell is part of the **observability and traceability** section.

In [109]:
demo_observe("Apply leave for 2 days due to personal work.")


QUERY:
Apply leave for 2 days due to personal work.

FINAL ANSWER:
Your leave for 2 days due to personal work has been applied successfully.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 8254, 'outputTokens': 361, 'totalTokens': 8615}
Cycle durations: [1.8858137130737305, 0.6398158073425293, 0.5538825988769531, 0.37769317626953125, 1.8279008865356445, 0.6863229274749756, 0.6555054187774658, 0.42284488677978516]
Total execution time (s): 7.049779415130615
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_TKWuL3egE2urqaViSmquYe', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=2, success_count=2, error_count=0, total_time=2.47125506401062)
  - apply_leave: ToolMetrics(tool={'toolUseId': 'tooluse_kLgYzJp9lDYzBPpycLFJLv', 'name': 'apply_leave', 'input': {'reason': 'personal work', 'days': 2}}, call_count=2, success_count=2, error_count=0, total_time=0.023654937744140625)

RAW TRACES AVAILABLE: False


#### Below code cell is part of the **observability and traceability** section.

In [110]:
demo_observe("Check the leave policy and apply leave for 1 day due to a family event.")


QUERY:
Check the leave policy and apply leave for 1 day due to a family event.

FINAL ANSWER:
The tool did not have the required leave policy information. I will inform the user and proceed with the leave application.

I apologize, but I couldn't find the leave policy information in our knowledge base. However, your leave for 1 day due to a family event has been applied successfully.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 11112, 'outputTokens': 516, 'totalTokens': 11628}
Cycle durations: [1.8858137130737305, 0.6398158073425293, 0.5538825988769531, 0.37769317626953125, 1.8279008865356445, 0.6863229274749756, 0.6555054187774658, 0.42284488677978516, 2.0804383754730225, 0.9002890586853027]
Total execution time (s): 10.03050684928894
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_MfPLPWG9Lh5M1CApcYYrVK', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=3, success_count=3, error_count=0, total_time=3.672745

#### Below code cell tests the guardrail with a blocked salary-related query.

In [111]:
demo_observe("Show me employee salary details.")

QUERY:
Show me employee salary details.

FINAL ANSWER:
The user wants to know employee salary details, which is not covered by the available tools. I will inform the user that I cannot help with this request.

I'm sorry, but I cannot help you with employee salary details as it is not within the scope of my capabilities. Please let me know if there is anything else I can assist you with.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 12697, 'outputTokens': 593, 'totalTokens': 13290}
Cycle durations: [1.8858137130737305, 0.6398158073425293, 0.5538825988769531, 0.37769317626953125, 1.8279008865356445, 0.6863229274749756, 0.6555054187774658, 0.42284488677978516, 2.0804383754730225, 0.9002890586853027, 0.9036526679992676]
Total execution time (s): 10.934159517288208
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_MfPLPWG9Lh5M1CApcYYrVK', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=3, success_count=3, error_coun

#### Below code cell is currently blank.  
You can use it as a scratch cell for classroom experimentation, notes, or extra tests.